# Predict New York City Airbnb Prices
**Student:** KEERTHI H  
**Project:** Linear Regression — Predict New York City Airbnb Prices

This notebook follows the attached AIML Minor Project 2 brief: EDA and auditing, missing-value handling, outlier treatment, feature selection and one-hot encoding, an 80/20 split, linear regression evaluation, and residual analysis.

**Dataset:** *New York City Airbnb Open Data* by AB_NYC on Kaggle. Download `AB_NYC_2019.csv` and upload it when prompted below.

## 1. Setup and load dataset
In Colab, upload `AB_NYC_2019.csv` when prompted. In Jupyter, place the CSV beside this notebook or change `DATA_PATH`.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
DATA_PATH = "AB_NYC_2019.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        csvs = [f for f in uploaded if f.lower().endswith(".csv")]
        if not csvs:
            raise FileNotFoundError("Upload AB_NYC_2019.csv.")
        DATA_PATH = csvs[0]
    except ImportError:
        raise FileNotFoundError("Place AB_NYC_2019.csv beside this notebook or set DATA_PATH.")

df_raw = pd.read_csv(DATA_PATH)
print("Dataset:", DATA_PATH, "| Shape:", df_raw.shape)
display(df_raw.head())


## 2. Step 1 — EDA and data auditing
Inspect shape, columns, types, variable groups, missingness percentages, duplicates, and summary statistics.

In [ ]:
print("Columns and dtypes:")
display(df_raw.dtypes.rename("dtype").to_frame())
numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_raw.select_dtypes(exclude=np.number).columns.tolist()
print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
missing_audit = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_audit)
print("Duplicate rows:", int(df_raw.duplicated().sum()))
display(df_raw.describe(include="all").T)


## 3. Step 2 — Cleaning and missing values
Drop `last_review` because it has substantial missingness and is not needed for this model. Set `reviews_per_month` to zero only when `number_of_reviews` is zero. Drop identifiers and high-cardinality `neighbourhood`; retain `neighbourhood_group` and `room_type`. Remaining feature missingness is imputed inside a pipeline fitted only on training data.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
df = df.drop_duplicates().copy()

if {"reviews_per_month", "number_of_reviews"}.issubset(df.columns):
    no_reviews = df["number_of_reviews"].fillna(0).eq(0)
    df.loc[no_reviews & df["reviews_per_month"].isna(), "reviews_per_month"] = 0

drop_cols = ["last_review", "id", "host_id", "name", "host_name",
             "neighbourhood", "listing_url", "host_url"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"]).copy()
df = df[df["price"] >= 0].copy()

print("Rows before cleaning:", len(df_raw))
print("Rows after duplicate/target cleaning:", len(df))
print("Rows removed:", len(df_raw) - len(df))
print("Remaining missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))


## 4. Step 3 — Outlier identification and treatment
Inspect price and minimum-night distributions. Then remove prices above the 95th percentile and minimum stays of 365 nights or more, as required by the brief.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
sns.histplot(df["price"], bins=60, ax=axes[0, 0])
axes[0, 0].set_title("Price Distribution Before Trimming")
sns.boxplot(x=df["price"], ax=axes[0, 1])
axes[0, 1].set_title("Price Boxplot Before Trimming")
sns.histplot(df["minimum_nights"], bins=50, ax=axes[1, 0])
axes[1, 0].set_title("Minimum Nights Before Trimming")
sns.boxplot(x=df["minimum_nights"], ax=axes[1, 1])
axes[1, 1].set_title("Minimum Nights Boxplot Before Trimming")
plt.tight_layout()
plt.show()

price_cap = df["price"].quantile(0.95)
rows_before_trim = len(df)
df = df[(df["price"] <= price_cap) & (df["minimum_nights"] < 365)].copy()
print(f"95th percentile price cap: ${price_cap:,.2f}")
print("Rows before outlier treatment:", rows_before_trim)
print("Rows after outlier treatment:", len(df))
print("Rows removed:", rows_before_trim - len(df))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["price"], bins=45, ax=axes[0])
axes[0].set_title("Price After Trimming")
axes[0].set_xlabel("Price (USD)")
sns.histplot(df["minimum_nights"], bins=40, ax=axes[1])
axes[1].set_title("Minimum Nights After Trimming")
plt.tight_layout()
plt.show()


## 5. Step 4 — Feature selection and encoding
Use numerical listing attributes plus `neighbourhood_group` and `room_type`. One-hot encode categorical features with `drop='first'` to avoid redundant dummy variables. The 200+ category neighbourhood field is excluded to keep the model simpler.

In [ ]:
target = "price"
categorical_features = [c for c in ["neighbourhood_group", "room_type"] if c in df.columns]
numeric_features = [c for c in [
    "latitude", "longitude", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count", "availability_365"
] if c in df.columns]
feature_columns = numeric_features + categorical_features
X = df[feature_columns].copy()
y = df[target].copy()
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target:", target)
display(X.head())


## 6. Step 5 — Train/test split
Use 80% of the data for training and 20% for testing, with `random_state=42` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 7. Step 6 — Linear regression and evaluation
Preprocessing is fitted inside the pipeline to avoid leakage. Evaluate R², MAE, MSE, and RMSE on the test set.

In [ ]:
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])
model = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
metrics = pd.DataFrame({
    "Metric": ["R-squared (R²)", "Mean Absolute Error (MAE)", "Mean Squared Error (MSE)", "Root Mean Squared Error (RMSE)"],
    "Test result": [r2, mae, mse, rmse],
    "Meaning": [
        "Variation in test prices explained by the model.",
        "Average absolute error in US dollars.",
        "Average squared error; large misses receive more weight.",
        "Error scale in dollars, sensitive to large misses."
    ]
})
display(metrics)
print(f"R² = {r2:.4f}")
print(f"MAE = ${mae:,.2f}")
print(f"MSE = {mse:,.2f}")
print(f"RMSE = ${rmse:,.2f}")


## 8. Step 7 — Residual analysis and interpretation
Compare actual and predicted prices, residual patterns, and absolute errors for cheaper versus more expensive listings.

In [ ]:
results = pd.DataFrame({
    "actual_price": y_test.to_numpy(),
    "predicted_price": y_pred
})
results["residual"] = results["actual_price"] - results["predicted_price"]
results["absolute_error"] = results["residual"].abs()

upper = max(results["actual_price"].max(), results["predicted_price"].max())
plt.figure(figsize=(7, 6))
sns.scatterplot(data=results, x="actual_price", y="predicted_price", alpha=0.45)
plt.plot([0, upper], [0, upper], "--", color="red", label="Perfect prediction")
plt.title("Actual vs Predicted Airbnb Prices")
plt.xlabel("Actual price (USD)")
plt.ylabel("Predicted price (USD)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=results, x="predicted_price", y="residual", alpha=0.45)
plt.axhline(0, linestyle="--", color="red")
plt.title("Residuals vs Predicted Prices")
plt.xlabel("Predicted price (USD)")
plt.ylabel("Residual: actual − predicted (USD)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
sns.histplot(results["residual"], bins=40, kde=True)
plt.title("Distribution of Prediction Errors")
plt.xlabel("Residual (USD)")
plt.tight_layout()
plt.show()

cutoff = results["actual_price"].median()
results["price_band"] = np.where(results["actual_price"] <= cutoff, "Cheaper half", "More expensive half")
error_by_band = results.groupby("price_band").agg(
    listings=("actual_price", "size"),
    MAE_USD=("absolute_error", "mean"),
    median_actual_price=("actual_price", "median")
).round(2)
display(error_by_band)
cheap_mae = error_by_band.loc["Cheaper half", "MAE_USD"]
expensive_mae = error_by_band.loc["More expensive half", "MAE_USD"]
if cheap_mae < expensive_mae:
    conclusion = "The model performs better on cheaper listings by absolute-dollar error."
elif cheap_mae > expensive_mae:
    conclusion = "The model performs better on the more expensive half by absolute-dollar error."
else:
    conclusion = "The model has equal mean absolute-dollar error in both price groups."
print(conclusion)
print("The result is specific to this test split; relative percentage error may give a different comparison.")


## 9. Business summary, limitations, and source
- **R²** measures how much variation in test prices the model explains; it is not the percentage of predictions that are correct.
- **MAE** is the average absolute prediction error in US dollars.
- **MSE/RMSE** penalize large errors more strongly than MAE.
- The model excludes the highest-priced 5% and minimum stays of 365 nights or more, so results do not describe those listings.
- A linear model with limited predictors cannot capture all price drivers, such as seasonality, amenities, demand, or special events.
- **Dataset source:** Kaggle, *New York City Airbnb Open Data* by AB_NYC, file `AB_NYC_2019.csv`. Download from Kaggle and retain attribution.